# Networked multi-microgrid reinforcement-learning experiments

This notebook documents the design and results of three GridAges experiment suites: centralized coordinated dispatch, official BenchMARL CTDE, and custom PyTorch CTDE. The report uses three random seeds (42, 43, and 44) and shows mean $\pm$ one standard deviation unless noted otherwise.

## Common environment and metrics

The power system contains three connected microgrids (MG1, MG2, and MG3) simulated with GridAges and pandapower. An episode contains 24 hourly decisions. The reported system metrics are episode return (higher is better), local-device operating cost (lower is better), summed safety violation (lower is better), and AC power-flow convergence rate (higher is better).

The reward used in these experiments is

$$R = -C - 10S,$$

where $C$ is operating cost and $S$ is the safety-violation measure. Therefore, return is not expected to converge to zero unless both quantities vanish. Power-flow convergence only means that pandapower solved the equations; it does not imply that voltage, loading, generator, or storage constraints were satisfied.

## Experiment-design comparison

| Experiment | Policy at execution | Information during training | Algorithms | Software |
|---|---|---|---|---|
| Coordinated dispatch | One controller observes the concatenated system state and outputs all MG actions | Global observation and joint action are inherent in the single controller | PPO, DDPG, SAC, TD3 | Stable-Baselines3 |
| BenchMARL CTDE | One local actor per MG; each actor uses its local observation | Centralized critic uses the grouped global state and, when required, joint actions | MAPPO, MADDPG, MASAC | Official BenchMARL implementations |
| Custom CTDE | One local actor per MG; each actor uses its local observation | Directly implemented centralized value/Q critics | MAPPO, MADDPG, MASAC | PyTorch |

In CTDE, critics are needed for training but actors alone generate actions at deployment. The coordinated-dispatch baseline instead requires global information during both training and execution.

In [ ]:
from pathlib import Path
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'examples' / 'multi_agent').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the GridAges repository.')

ROOT = find_repo_root()
REPORT = ROOT / 'examples' / 'multi_agent' / 'experiment_report'
metrics = pd.read_csv(REPORT / 'final_metrics.csv')
metrics

## 1. Centralized coordinated dispatch

The coordinated-dispatch baseline flattens and concatenates the three local observations into one observation and concatenates the three action vectors into one joint action. A single Stable-Baselines3 policy controls all microgrids. The environment uses the original networked-microgrid setup, observation and reward normalization, deterministic evaluation, 500,000 training steps, and three seeds.

![Centralized coordinated-dispatch results](assets/coordinated_dispatch_comparison.png)

PPO achieved the best and most stable final return ($-88.2 \pm 10.5$). SAC and TD3 were less effective but remained reasonably stable. DDPG showed severe seed sensitivity, reflected by its large return, cost, and safety error bars.

In [ ]:
metrics.loc[metrics['experiment'] == 'Coordinated dispatch', [
    'algorithm', 'return_mean', 'return_std', 'operating_cost_mean',
    'safety_mean', 'convergence_mean'
]].reset_index(drop=True)

## 2. Official BenchMARL CTDE

This suite adapts GridAges to BenchMARL/TorchRL and uses the official MAPPO, MADDPG, and MASAC implementations. All microgrids form one agent group. Actors use padded local observations and normalized local actions; centralized critics can use the global state. Rewards are shared, actor architectures are equal with independent actor parameters, and the microgrids use heterogeneous load, DG, ESS, and renewable parameters. Runs target 500,000 frames; because the 240-frame collection batch determines the evaluation schedule, the last common plotted evaluation is at 493,920 frames.

![Official BenchMARL CTDE results](assets/benchmarl_comparison.png)

MAPPO was the most stable BenchMARL method, ending at $-303.4 \pm 5.7$. MADDPG minimized local operating cost but produced very large safety violations, so its return remained poor. MASAC improved but retained substantial cross-seed variability.

In [ ]:
metrics.loc[metrics['experiment'] == 'BenchMARL CTDE', [
    'algorithm', 'return_mean', 'return_std', 'operating_cost_mean',
    'safety_mean', 'convergence_mean'
]].reset_index(drop=True)

## 3. Custom PyTorch CTDE

The custom suite directly implements separate local actors and centralized critics in PyTorch. MAPPO uses centralized value critics; MADDPG and MASAC use centralized action-value critics that receive joint observations and actions. These runs use shared rewards, homogeneous neural-network layer layouts with independent actor weights, heterogeneous microgrid parameters, 500,000 steps, five deterministic evaluation episodes per checkpoint, and three seeds.

![Custom PyTorch CTDE results](assets/ctde_comparison.png)

MAPPO and MADDPG achieved similar final returns ($-97.9 \pm 20.6$ and $-95.4 \pm 19.0$). MADDPG obtained the lowest local operating cost, while MAPPO had the lowest mean safety violation. MASAC remained substantially worse and more variable.

In [ ]:
metrics.loc[metrics['experiment'] == 'Custom CTDE', [
    'algorithm', 'return_mean', 'return_std', 'operating_cost_mean',
    'safety_mean', 'convergence_mean'
]].reset_index(drop=True)

## Interpretation and limitations

The strongest within-suite conclusions are: centralized PPO was the best Stable-Baselines3 baseline; official BenchMARL MAPPO was the most stable BenchMARL method; and custom MAPPO/MADDPG performed similarly after 500,000 steps. All methods reported a power-flow convergence rate of 1.0, but their safety scores differed materially.

Absolute returns across the three suites should **not** be interpreted as a strict common leaderboard. The coordinated-dispatch experiments use the original homogeneous microgrid setup, whereas the BenchMARL and custom CTDE experiments use the heterogeneous configuration. The implementations also differ in normalization, batching, exploration, evaluation interval, and neural-network details. A controlled architecture study should rerun all suites with the same environment configuration, fixed held-out evaluation days, identical evaluation episodes, and matched model capacity.

Two environment limitations are especially important: (1) the reported operating cost includes local ESS/DG device costs but not a complete DSO import/export settlement cost, and (2) successive evaluations currently advance through test days, so curve variation includes both policy changes and scenario changes. These should be addressed before treating the numbers as publication-ready.

## Reproducing the report figures

Raw logs are intentionally ignored by Git. From the repository root, regenerate each source figure with the plotting script in its experiment folder. The four-panel CTDE-style plots can be generated with:

```bash
python examples/multi_agent/benchmarl_networked_mgs/plot_results.py \
  --log-dir examples/multi_agent/benchmarl_networked_mgs/logs \
  --output examples/multi_agent/benchmarl_networked_mgs/logs/benchmarl_comparison.png

python examples/multi_agent/benchmarl_networked_mgs/plot_results.py \
  --log-dir examples/multi_agent/ctde_marl_networked_mgs/logs/ctde_500k \
  --output examples/multi_agent/ctde_marl_networked_mgs/logs/ctde_500k/ctde_comparison_curves.png \
  --title 'GridAges CTDE evaluation (mean across seeds)'
```

After regenerating, copy only the final PNG files into `experiment_report/assets/`; do not commit logs or checkpoints.